In [1]:
import pandas as pd
import numpy as np

# Load raw data
df = pd.read_csv("monopoly_data_full.csv")

# Convert timestamp
df["when_datetime_ist"] = pd.to_datetime(
    df["when_datetime_ist"],
    errors="coerce"
)

# Sort chronologically
df = df.sort_values(
    "when_datetime_ist"
).reset_index(drop=True)

print("Raw shape:", df.shape)

Raw shape: (50000, 11)


Remove exact duplicate rows

In [2]:
df = df.drop_duplicates().reset_index(drop=True)

print("After removing exact duplicates:", df.shape)

After removing exact duplicates: (49981, 11)


Identify completed rounds

In [3]:
#let's investigate chance_multiplier

print(
    df["chance_multiplier"]
    .value_counts(dropna=False)
    .sort_index()
)

chance_multiplier
0     48816
2       320
3       149
4       189
5       160
8       191
10      156
Name: count, dtype: int64


In [4]:
completed = df[
    df["chance_multiplier"].fillna(0) == 0
].copy()

print("Completed records:", len(completed))
print(
    "Unique round codes:",
    completed["round_code"].nunique()
)

Completed records: 48816
Unique round codes: 48816


Check whether round codes are now unique

In [5]:
duplicate_completed = completed[
    completed["round_code"].duplicated(keep=False)
]

print(
    "Duplicate completed round codes:",
    len(duplicate_completed)
)

if len(duplicate_completed) > 0:
    print(
        duplicate_completed[
            [
                "round_code",
                "when_datetime_ist",
                "result",
                "multiplier",
                "chance_multiplier"
            ]
        ].head(20)
    )

Duplicate completed round codes: 0


Remove useless columns

In [6]:
completed = completed.drop(
    columns=["video_uid", "when"],
    errors="ignore"
)

Create useful time features

In [7]:
completed["date"] = (
    completed["when_datetime_ist"].dt.date
)

completed["hour"] = (
    completed["when_datetime_ist"].dt.hour
)

completed["minute"] = (
    completed["when_datetime_ist"].dt.minute
)

completed["second"] = (
    completed["when_datetime_ist"].dt.second
)

completed["day_of_week"] = (
    completed["when_datetime_ist"].dt.dayofweek
)

completed["is_weekend"] = (
    completed["day_of_week"] >= 5
).astype(int)

Calculate time between completed rounds

In [8]:
completed = completed.sort_values(
    "when_datetime_ist"
).reset_index(drop=True)

completed["seconds_since_previous"] = (
    completed["when_datetime_ist"]
    .diff()
    .dt.total_seconds()
)

print(
    completed["seconds_since_previous"]
    .describe()
)

count    48815.000000
mean        57.692635
std         89.268349
min         32.000000
25%         40.000000
50%         43.000000
75%         46.000000
max      11554.000000
Name: seconds_since_previous, dtype: float64


Look for:

negative values

zero values

extremely large gaps

unusual intervals

In [9]:
print(
    "Negative gaps:",
    (completed["seconds_since_previous"] < 0).sum()
)

print(
    "Zero gaps:",
    (completed["seconds_since_previous"] == 0).sum()
)

print(
    "Gaps > 5 minutes:",
    (completed["seconds_since_previous"] > 300).sum()
)

Negative gaps: 0
Zero gaps: 0
Gaps > 5 minutes: 188


Validate the target

In [10]:
print(
    completed["result"]
    .value_counts()
    .sort_index()
)

result
1     20313
2     13980
5      6561
10     3698
20     2751
40      851
50      662
Name: count, dtype: int64


In [11]:
result_distribution = (
    completed["result"]
    .value_counts(normalize=True)
    .sort_index() * 100
)

print(
    result_distribution.round(2)
)

result
1     41.61
2     28.64
5     13.44
10     7.58
20     5.64
40     1.74
50     1.36
Name: proportion, dtype: float64


In [12]:
#Create a clean csv
completed.to_csv(
    "monopoly_cleaned.csv",
    index=False
)

print("Saved: monopoly_cleaned.csv")
print("Final shape:", completed.shape)

Saved: monopoly_cleaned.csv
Final shape: (48816, 16)


Feature Engineering

In [13]:
#Keep only completed rounds

df = df[
    df["chance_multiplier"].fillna(0) == 0
].copy()

df = df.reset_index(drop=True)

print("Completed rounds:", len(df))
print("Unique round codes:", df["round_code"].nunique())


Completed rounds: 48816
Unique round codes: 48816


In [14]:
#sort again after filtering

df = df.sort_values(
    "when_datetime_ist"
).reset_index(drop=True)

In [15]:
#inspect the first and last records:
print(
    df[
        [
            "round_code",
            "when_datetime_ist",
            "result",
            "multiplier"
        ]
    ].head()
)

print()

print(
    df[
        [
            "round_code",
            "when_datetime_ist",
            "result",
            "multiplier"
        ]
    ].tail()
)

     round_code         when_datetime_ist  result  multiplier
0  mdHdhY2_ezW8 2026-07-30 01:25:09+05:30      10          10
1  mdHdhY2_ezWU 2026-07-30 01:25:51+05:30       1           1
2  mdHdhY2_ezXg 2026-07-30 01:26:33+05:30       2           2
3  mdHdhY2_ezXI 2026-07-30 01:27:16+05:30       5           5
4  mdHdhY2_ezUg 2026-07-30 01:27:59+05:30       1           1

         round_code         when_datetime_ist  result  multiplier
48811  mdHdhY2_FMK4 2026-08-31 15:40:22+05:30       2           2
48812  mdHdhY2_FMKU 2026-08-31 15:41:04+05:30       1           1
48813  mdHdhY2_FML4 2026-08-31 15:41:40+05:30       5           5
48814  mdHdhY2_FMLU 2026-08-31 15:42:18+05:30       1           1
48815  mdHdhY2_FMLA 2026-08-31 15:42:55+05:30      10          10


Create the target

In [16]:
df["target"] = df["result"]

In [17]:
#create lag features

for lag in range(1, 6):
    df[f"result_lag_{lag}"] = df["result"].shift(lag)


In [18]:
print(
    df[
        [
            "result",
            "result_lag_1",
            "result_lag_2",
            "result_lag_3",
            "result_lag_4",
            "result_lag_5"
        ]
    ].head(10)
)

   result  result_lag_1  result_lag_2  result_lag_3  result_lag_4  \
0      10           NaN           NaN           NaN           NaN   
1       1          10.0           NaN           NaN           NaN   
2       2           1.0          10.0           NaN           NaN   
3       5           2.0           1.0          10.0           NaN   
4       1           5.0           2.0           1.0          10.0   
5       2           1.0           5.0           2.0           1.0   
6       2           2.0           1.0           5.0           2.0   
7       5           2.0           2.0           1.0           5.0   
8       1           5.0           2.0           2.0           1.0   
9       2           1.0           5.0           2.0           2.0   

   result_lag_5  
0           NaN  
1           NaN  
2           NaN  
3           NaN  
4           NaN  
5          10.0  
6           1.0  
7           2.0  
8           5.0  
9           1.0  


create previous multiplier feature

In [19]:
for lag in range(1, 6):
    df[f"multiplier_lag_{lag}"] = (
        df["multiplier"].shift(lag)
    )

Rolling statistics

In [20]:
#rolling average multiplier

df["multiplier_mean_5"] = (
    df["multiplier"]
    .shift(1)
    .rolling(5)
    .mean()
)

df["multiplier_mean_10"] = (
    df["multiplier"]
    .shift(1)
    .rolling(10)
    .mean()
)

df["multiplier_mean_20"] = (
    df["multiplier"]
    .shift(1)
    .rolling(20)
    .mean()
)

Rolling multiplier volatility

In [21]:
df["multiplier_std_5"] = (
    df["multiplier"]
    .shift(1)
    .rolling(5)
    .std()
)

df["multiplier_std_10"] = (
    df["multiplier"]
    .shift(1)
    .rolling(10)
    .std()
)

df["multiplier_std_20"] = (
    df["multiplier"]
    .shift(1)
    .rolling(20)
    .std()
)

Create recent outcomes

In [22]:
results = [1, 2, 5, 10, 20, 40, 50]

for result_value in results:
    df[f"result_{result_value}_count_10"] = (
        (df["result"] == result_value)
        .shift(1)
        .rolling(10)
        .sum()
    )

In [23]:
#for the previous 20 rounds:



for result_value in results:
    df[f"result_{result_value}_count_20"] = (
        (df["result"] == result_value)
        .shift(1)
        .rolling(20)
        .sum()
    )

Create time features

In [24]:
df["hour"] = (
    df["when_datetime_ist"].dt.hour
)

df["minute"] = (
    df["when_datetime_ist"].dt.minute
)

df["day_of_week"] = (
    df["when_datetime_ist"].dt.dayofweek
)

df["is_weekend"] = (
    df["day_of_week"] >= 5
).astype(int)

In [25]:
#We could also use cyclical encoding for hour:
df["hour_sin"] = np.sin(
    2 * np.pi * df["hour"] / 24
)

df["hour_cos"] = np.cos(
    2 * np.pi * df["hour"] / 24
)

This represents the fact that:

23:59
  ->
00:00

are actually close in time.

Create a streak feature

In [26]:
#Let's calculate how many consecutive times the previous result was the same.

def calculate_streak(series):
    streaks = []
    previous = None
    streak = 0

    for value in series:
        if value == previous:
            streak += 1
        else:
            streak = 1

        streaks.append(streak)
        previous = value

    return pd.Series(
        streaks,
        index=series.index
    )

df["result_streak"] = calculate_streak(
    df["result"]
)

# We need the streak BEFORE the current round
df["previous_streak"] = (
    df["result_streak"].shift(1)
)

Create the final modeling dataset

In [27]:
#define which columns the model is allowed to see
feature_columns = [
    "result_lag_1",
    "result_lag_2",
    "result_lag_3",
    "result_lag_4",
    "result_lag_5",

    "multiplier_lag_1",
    "multiplier_lag_2",
    "multiplier_lag_3",
    "multiplier_lag_4",
    "multiplier_lag_5",

    "multiplier_mean_5",
    "multiplier_mean_10",
    "multiplier_mean_20",

    "multiplier_std_5",
    "multiplier_std_10",
    "multiplier_std_20",

    "hour",
    "minute",
    "day_of_week",
    "is_weekend",

    "hour_sin",
    "hour_cos",

    "previous_streak"
]

In [28]:
#add the rolling result counts

for result_value in results:
    feature_columns.append(
        f"result_{result_value}_count_10"
    )

    feature_columns.append(
        f"result_{result_value}_count_20"
    )

In [29]:
model_df = df[
    feature_columns + ["target"]
].copy()

Remove rows without enough history

In [30]:
#The first 20-ish rounds don't have enough historical information to calculate all of our features.
model_df = model_df.dropna().reset_index(drop=True)

print("Final modeling shape:", model_df.shape)



Final modeling shape: (48796, 38)


In [31]:
print(
    model_df.head()
)

   result_lag_1  result_lag_2  result_lag_3  result_lag_4  result_lag_5  \
0           1.0           1.0          10.0           1.0           2.0   
1           2.0           1.0           1.0          10.0           1.0   
2           1.0           2.0           1.0           1.0          10.0   
3           2.0           1.0           2.0           1.0           1.0   
4           1.0           2.0           1.0           2.0           1.0   

   multiplier_lag_1  multiplier_lag_2  multiplier_lag_3  multiplier_lag_4  \
0               1.0               1.0              10.0               1.0   
1               2.0               1.0               1.0              10.0   
2               1.0               2.0               1.0               1.0   
3               2.0               1.0               2.0               1.0   
4               1.0               2.0               1.0               2.0   

   multiplier_lag_5  ...  result_5_count_20  result_10_count_10  \
0               2.0

Check the target distribution again

In [32]:
print(
    model_df["target"]
    .value_counts()
    .sort_index()
)

print()

print(
    (
        model_df["target"]
        .value_counts(normalize=True)
        .sort_index() * 100
    ).round(2)
)

target
1     20307
2     13973
5      6559
10     3694
20     2750
40      851
50      662
Name: count, dtype: int64

target
1     41.62
2     28.64
5     13.44
10     7.57
20     5.64
40     1.74
50     1.36
Name: proportion, dtype: float64


save the feature data

In [33]:
model_df.to_csv(
    "monopoly_features.csv",
    index=False
)

print("Saved successfully!")

Saved successfully!
